# PEPE/WETH: offline audit
Run from this article directory. No network or credentials are required. The packed RPC snapshot is the source.
The comparison is conditional on fixed inputs and selected historical sequences; it is not a population prevalence or net-profit estimate.

In [1]:
import json, subprocess, sys
from pathlib import Path
root = Path.cwd()
assert (root / "analyze.py").exists(), "Open the notebook in the article directory"
def run_checked(*arguments):
    result = subprocess.run([sys.executable, *arguments], capture_output=True, text=True)
    if result.returncode:
        print(result.stdout)
        print(result.stderr, file=sys.stderr)
    result.check_returncode()
run_checked("validate.py", "--source-only")
run_checked("analyze.py", "--source", "archive")
run_checked("validate.py")
json.loads((root / "data/validation.json").read_text())

{'snapshot_sha256': 'pass',
 'snapshot_records': 10349,
 'complete_event_range': [17070000, 17079999],
 'duplicate_events': 0,
 'reserve_transitions_checked': 25768,
 'selected_receipts_checked': 231,
 'independent_fraction_counterfactuals': 115,
 'no_intervening_flow_controls': 58,
 'second_provider_receipts_checked': 3,
 'headline_reconciliation': 'pass',
 'inventory_closure_base_units': 'exact',
 'limitations': ['RPC responses are not independently verified against Ethereum receipt trie roots.',
  'No mempool history, common-owner attribution, or private builder payments are available.',
  'The counterfactual holds inputs and routing fixed; it is not a full market simulation.'],
 'negative_pilot': {'events': 1476, 'swaps': 734, 'structural_candidates': 0}}

In [2]:
from fractions import Fraction
import csv
v = next(csv.DictReader((root / "data/victims.csv").open()))
r0, r1, x = map(int, [v["counterfactual_reserve0_raw"], v["counterfactual_reserve1_raw"], v["input_raw"]])
effective = Fraction(997, 1000) * x
counterfactual = Fraction(r0) - Fraction(r0 * r1) / (r1 + effective)
assert counterfactual.numerator // counterfactual.denominator == int(v["counterfactual_output_raw"])
{k: v[k] for k in ["tx_hash", "input_weth", "actual_pepe", "counterfactual_pepe", "shortfall_bps"]}

{'tx_hash': '0x9491232feccf49b65a79e172826f15eb4d716c6887ab6f732f41f1b1d03e4dcd',
 'input_weth': '3.610090956873247782',
 'actual_pepe': '91125518089.315649730676525978',
 'counterfactual_pepe': '91958534765.7826439540174926',
 'shortfall_bps': '90.58611890551304996241973068191965436644715401490992739507633895892625'}